# Dense WDM runtime optimisation benchmark

This notebook is the controlled performance laboratory for the full-year, six-link WDM one-arm response. It deliberately does **not** replace the scientific validation notebooks.

The frozen physical problem is:

- $N=3{,}200{,}000$, $dt=10\,\mathrm{s}$, $N_f=100$;
- `calc_m0=False`, $L_{\rm trunc}=25$;
- production `complex64` CPU assembly;
- current reference tuning: `row_chunk_size=32640`, `lag_block_size=17`;
- current link schedule: groups of 2 links on 3 workers.

Reference measurements on `calx038` were approximately 0.124 s for the paired DWT stage, 0.328 s for the selected six-link shift, and 0.453 s for complete A1. Treat those values as a sanity check, not hard pass/fail thresholds.

Run this notebook from a freshly restarted kernel. It tunes one dimension at a time, checks every output against a frozen link map, and exports JSON/CSV records. Only winners should proceed to the existing end-to-end response/mismatch validation.

## 0. CPU environment and imports

The backend variables must be set before importing JAX, FEW, WDM, or the response package.

In [ ]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = ""


In [ ]:
import gc
import json
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path
        for path in [ROOT, *ROOT.parents]
        if (path / "WDM_LISAresponse").exists()
        and (path / "WDM_GW_wavelets").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Run from a directory beneath a parent containing both "
        "WDM_LISAresponse and WDM_GW_wavelets."
    )

WDM_LISA_ROOT = PROJECT_ROOT / "WDM_LISAresponse"
WDM_GW_ROOT = PROJECT_ROOT / "WDM_GW_wavelets"
BENCHMARK_DIR = WDM_LISA_ROOT / "testing_optimisation"

for source_path in (
    WDM_LISA_ROOT / "src",
    WDM_GW_ROOT / "src",
    BENCHMARK_DIR,
):
    source_text = str(source_path)
    if source_text not in sys.path:
        sys.path.insert(0, source_text)

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

import WDM
import WDM_LISAresponse
from few.waveform import GenerateEMRIWaveform
from lisaorbits import InterpolatedOrbits
from lisaorbits.file import read_orbit_file

from benchmark_runtime_helpers import (
    benchmark_candidate,
    block_result,
    compare_arrays,
    compare_link_maps,
    fastest_passing,
    git_state,
    memory_snapshot,
    numerical_gate,
    save_json,
    system_metadata,
    timed_call,
)
from WDM_LISAresponse.code.lisa_response.one_arm_plan import OneArmWDMPlan
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_context_from_wdm,
    _build_wdm_dT_grid,
)
from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)

if jax.default_backend() != "cpu":
    raise RuntimeError(f"Expected the CPU backend, got {jax.default_backend()!r}.")


## 1. Experiment controls

`smoke` verifies that the notebook works, `quick` is suitable while developing, and `full` is the result-producing run. The sweep is intentionally staged rather than Cartesian: row size first, then lag block size at the winning row size, then link scheduling on the winning kernel configuration.

In [ ]:
RUN_MODE = "quick"  # "smoke", "quick", or "full"

RUN_PROFILES = {
    "smoke": {
        "warmups": 0,
        "repeats": 1,
        "a1_repeats": 1,
        "row_sizes": [32_000, 32_640],
        "lag_sizes": [17, 51],
    },
    "quick": {
        "warmups": 1,
        "repeats": 5,
        "a1_repeats": 3,
        "row_sizes": [8_192, 16_000, 32_000, 32_640],
        "lag_sizes": [8, 16, 17, 25, 51],
    },
    "full": {
        "warmups": 2,
        "repeats": 11,
        "a1_repeats": 9,
        "row_sizes": [4_096, 8_192, 16_000, 24_000, 32_000, 32_640],
        "lag_sizes": [3, 8, 16, 17, 25, 51],
    },
}
PROFILE = RUN_PROFILES[RUN_MODE]

N = 3_200_000
DT = 10.0
NF = 100
Q_WDM = 4
D_WDM = 4
A_FRAC = 0.25
L_TRUNC = 25
CALC_M0 = False

BASE_ROW_CHUNK_SIZE = 32_640
BASE_LAG_BLOCK_SIZE = 17
BASE_GROUP_SIZE = 2
BASE_WORKERS = 3

BASE_SHIFT_KWARGS = {
    "assembly_backend": "production",
    "assembly_precision": "complex64",
    "row_chunk_size": BASE_ROW_CHUNK_SIZE,
    "lag_block_size": BASE_LAG_BLOCK_SIZE,
    "batch_chunk": 6,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
}

SCHEDULES = [
    {"name": "materialized", "mode": "materialized", "group_size": 6, "workers": 1},
    {"name": "groups_3_workers_2", "mode": "materialized_parallel_links", "group_size": 3, "workers": 2},
    {"name": "groups_2_workers_3", "mode": "materialized_parallel_links", "group_size": 2, "workers": 3},
    {"name": "groups_1_workers_6", "mode": "materialized_parallel_links", "group_size": 1, "workers": 6},
]

REL_L2_TOLERANCE = 1.0e-6
MAX_ABS_TOLERANCE = None
MIN_MEANINGFUL_SPEEDUP = 1.05

REUSE_WAVEFORM_CACHE = True
REUSE_WDM_CACHE = True
RUN_DWT_BENCHMARK = True
RUN_COMPLETE_A1 = True
SAVE_RESULTS = True
STRICT_PROVENANCE = False
EXPECTED_WDM_COMMIT_PREFIX = "a32028bf"

# Add one entry after implementing a new exact backend or feature switch.
# It will be compared against the same frozen link output, but it is not
# automatically promoted into the staged row/lag/schedule winner.
CUSTOM_EXPERIMENTS = [
    # {
    #     "name": "production_cached_phase",
    #     "shift_overrides": {"assembly_backend": "production_cached_phase"},
    #     "mode": "materialized",
    #     "group_size": 6,
    #     "workers": 1,
    # },
]


## 2. Provenance and fixed physical inputs

A dirty checkout is recorded rather than hidden. Set `STRICT_PROVENANCE=True` only after the intended WDM commit is pinned and the worktrees are clean.

In [ ]:
HOST = system_metadata()
WDM_STATE = git_state(WDM.__file__)
RESPONSE_STATE = git_state(WDM_LISAresponse.__file__)

print(json.dumps({"host": HOST, "WDM": WDM_STATE, "response": RESPONSE_STATE}, indent=2))

wdm_commit = WDM_STATE.get("commit") or ""
provenance_ok = wdm_commit.startswith(EXPECTED_WDM_COMMIT_PREFIX)
if not provenance_ok:
    message = (
        f"WDM commit {wdm_commit or 'unknown'} does not match the expected "
        f"prefix {EXPECTED_WDM_COMMIT_PREFIX!r}."
    )
    if STRICT_PROVENANCE:
        raise RuntimeError(message)
    warnings.warn(message)

if WDM_STATE.get("dirty") or RESPONSE_STATE.get("dirty"):
    message = "At least one benchmark checkout is dirty; the JSON record will preserve this fact."
    if STRICT_PROVENANCE:
        raise RuntimeError(message)
    warnings.warn(message)

ORBIT_FILE = WDM_LISA_ROOT / "examples" / "esa-trailing-orbits.h5"
if not ORBIT_FILE.exists():
    raise FileNotFoundError(ORBIT_FILE)

orbit_data = read_orbit_file(str(ORBIT_FILE))
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)
YEAR_SI = 365.25 * 24.0 * 3600.0
t0 = float(orbit_data.t0 + YEAR_SI)

THETA = np.pi / 2
PHI = 0.0
print("Initial memory:", memory_snapshot())


## 3. Reuse or create the full-year waveform cache

The paths deliberately match notebook 03 so the existing `calx038` cache is reused. Set the reuse flags to `False` after changing waveform generation, sampling, padding, or WDM tiling.

In [ ]:
CACHE_DIR = WDM_LISA_ROOT / "examples"
WAVEFORM_CACHE = CACHE_DIR / "_local_wdm_03_waveform_cache.npz"
WDM_CACHE = CACHE_DIR / "_local_wdm_03_wdm_coefficients.npz"
CACHE_METADATA = CACHE_DIR / "_local_wdm_03_cache_metadata.json"

if REUSE_WAVEFORM_CACHE and WAVEFORM_CACHE.exists():
    with np.load(WAVEFORM_CACHE, allow_pickle=False) as cached:
        times_pad = np.asarray(cached["times_pad"])
        h_plus = np.asarray(cached["h_plus"])
        h_cross = np.asarray(cached["h_cross"])
    waveform_generation_s = 0.0
    print("Reused:", WAVEFORM_CACHE)
else:
    td_gen = GenerateEMRIWaveform(
        "FastKerrEccentricEquatorialFlux",
        sum_kwargs={"pad_output": True, "odd_len": True},
        return_list=True,
        force_backend="cpu",
    )
    model_kwargs = {
        "td_gen": td_gen,
        "m1": 1.0e6,
        "m2": 10.0,
        "a": 0.9,
        "p0": 8.5,
        "e0": 0.3,
        "x0": 1.0,
        "dist": 1.0,
        "theta": THETA,
        "phi": PHI,
        "qK": np.pi / 3,
        "phiK": np.pi / 3,
        "Phi_phi0": np.pi / 3,
        "Phi_theta0": 0.0,
        "Phi_r0": np.pi / 3,
        "eps": 1.0e-2,
    }

    waveform_payload, waveform_generation_s = timed_call(
        lambda: generate_windowed_padded_waveform(
            waveform_model="emri",
            t0=t0,
            dt=DT,
            N=N,
            model_kwargs=model_kwargs,
            window_function="tukey",
            window_kwargs={"alpha": 0.005, "tukey_factor": 4.0},
            wdm_Nf=NF,
            return_padding_info=True,
        )
    )
    times_pad, h_plus, h_cross, _padding = waveform_payload
    np.savez(
        WAVEFORM_CACHE,
        times_pad=times_pad,
        h_plus=h_plus,
        h_cross=h_cross,
    )
    print("Created:", WAVEFORM_CACHE)

if times_pad.size != N:
    raise ValueError(f"Expected {N} padded samples, found {times_pad.size}.")

print("Waveform generation [s]:", waveform_generation_s)
print("Waveform host memory [GiB]:", (times_pad.nbytes + h_plus.nbytes + h_cross.nbytes) / 1024**3)
print("Memory:", memory_snapshot())


## 4. Build the WDM grid and frozen production plan

In [ ]:
grid_context, grid_build_s = timed_call(
    lambda: _build_wdm_dT_grid(
        times_pad,
        Nf=NF,
        d=D_WDM,
        A_frac=A_FRAC,
        q=Q_WDM,
        calc_m0=CALC_M0,
    )
)

def build_one_arm_plan(shift_overrides=None):
    shift_kwargs = {**BASE_SHIFT_KWARGS, **(shift_overrides or {})}
    return OneArmWDMPlan.build_from_context(
        grid_context,
        orbits,
        THETA,
        PHI,
        L_trunc=L_TRUNC,
        variable_shift_kwargs=shift_kwargs,
    )

baseline_plan, baseline_plan_build_s = timed_call(build_one_arm_plan)
expected_shape = (grid_context["wdm"].Nt, grid_context["wdm"].Nf)

print("WDM shape:", expected_shape)
print("Grid build [s]:", grid_build_s)
print("Baseline plan build [s]:", baseline_plan_build_s)
print("Baseline plan [MiB]:", baseline_plan.shift_plan.device_plan_memory_bytes / 1024**2)
print("Memory:", memory_snapshot())


## 5. Reuse or create WDM coefficient cache, then benchmark DWTs

The shift sweeps use one fixed pair of WDM coefficient arrays. Complete A1 below recomputes the transforms inside the timed callable.

In [ ]:
def sequential_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        h_plus,
        h_cross,
        grid_context["wdm"],
        parallel_wdm_transforms=False,
    )

def parallel_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        h_plus,
        h_cross,
        grid_context["wdm"],
        parallel_wdm_transforms=True,
    )

dwt_results = {}
if RUN_DWT_BENCHMARK:
    sequential_context, sequential_row = benchmark_candidate(
        "sequential_dwt",
        sequential_dwt,
        warmups=PROFILE["warmups"],
        repeats=PROFILE["repeats"],
        m_start=1,
    )
    parallel_context, parallel_row = benchmark_candidate(
        "parallel_dwt",
        parallel_dwt,
        warmups=PROFILE["warmups"],
        repeats=PROFILE["repeats"],
        m_start=1,
    )
    dwt_comparison = {
        "plus": compare_arrays(parallel_context["w_t_plus"], sequential_context["w_t_plus"], m_start=1),
        "cross": compare_arrays(parallel_context["w_t_cross"], sequential_context["w_t_cross"], m_start=1),
    }
    dwt_results = {
        "sequential": sequential_row,
        "parallel": parallel_row,
        "comparison": dwt_comparison,
    }
    print(pd.DataFrame([sequential_row, parallel_row])[["name", "first_call_s", "median_s", "iqr_s"]])
    print("DWT comparison:", dwt_comparison)
else:
    parallel_context = parallel_dwt()
    block_result(parallel_context)

cache_is_compatible = False
if REUSE_WDM_CACHE and WDM_CACHE.exists():
    with np.load(WDM_CACHE, allow_pickle=False) as cached:
        cached_plus = np.asarray(cached["w_plus"])
        cached_cross = np.asarray(cached["w_cross"])
        cached_nf = int(np.asarray(cached["Nf"]).item())
        cached_calc_m0 = bool(np.asarray(cached["calc_m0"]).item())
    cache_is_compatible = (
        cached_plus.shape == expected_shape
        and cached_cross.shape == expected_shape
        and cached_nf == NF
        and cached_calc_m0 == CALC_M0
    )

if cache_is_compatible:
    w_plus_host = cached_plus
    w_cross_host = cached_cross
    print("Reused:", WDM_CACHE)
else:
    w_plus_host = np.asarray(parallel_context["w_t_plus"])
    w_cross_host = np.asarray(parallel_context["w_t_cross"])
    np.savez(
        WDM_CACHE,
        times_pad=times_pad,
        w_plus=w_plus_host,
        w_cross=w_cross_host,
        Nf=np.asarray(NF),
        q_wdm=np.asarray(Q_WDM),
        d_wdm=np.asarray(D_WDM),
        A_frac=np.asarray(A_FRAC),
        calc_m0=np.asarray(CALC_M0),
    )
    print("Created:", WDM_CACHE)

if w_plus_host.shape != expected_shape or w_cross_host.shape != expected_shape:
    raise ValueError("Cached WDM coefficient shape does not match the active grid.")

w_plus = jnp.asarray(w_plus_host)
w_cross = jnp.asarray(w_cross_host)
block_result((w_plus, w_cross))
print("Coefficient dtype:", w_plus.dtype)
print("Coefficient device memory [MiB]:", (w_plus.nbytes + w_cross.nbytes) / 1024**2)


## 6. Freeze the current six-link reference

Every later candidate is compared with this output. Timing uses the currently validated groups-of-two/three-worker schedule; the first call is reported separately from steady-state samples.

In [ ]:
def apply_plan(plan, *, mode, group_size, workers, plus=w_plus, cross=w_cross):
    kwargs = {"execution_mode": mode}
    if mode == "materialized_parallel_links":
        kwargs.update(
            parallel_links_per_group=int(group_size),
            parallel_max_workers=int(workers),
        )
    return plan.apply_device(plus, cross, **kwargs)

reference_links, baseline_shift_row = benchmark_candidate(
    "current_groups_2_workers_3",
    lambda: apply_plan(
        baseline_plan,
        mode="materialized_parallel_links",
        group_size=BASE_GROUP_SIZE,
        workers=BASE_WORKERS,
    ),
    warmups=PROFILE["warmups"],
    repeats=PROFILE["repeats"],
    m_start=1,
    metadata={
        "row_chunk_size": BASE_ROW_CHUNK_SIZE,
        "lag_block_size": BASE_LAG_BLOCK_SIZE,
        "group_size": BASE_GROUP_SIZE,
        "workers": BASE_WORKERS,
        "execution_mode": "materialized_parallel_links",
    },
)
print(pd.Series({key: value for key, value in baseline_shift_row.items() if key not in {"all_s", "per_link"}}))


## 7. Common plan-candidate runner

Each temporary plan is built, benchmarked, compared, and released before the next candidate. `plan_build_s` is recorded but excluded from repeated-application timing.

In [ ]:
def evaluate_plan_candidate(
    *,
    name,
    shift_overrides,
    mode="materialized",
    group_size=6,
    workers=1,
):
    candidate_plan = None
    candidate_output = None
    try:
        candidate_plan, plan_build_s = timed_call(
            lambda: build_one_arm_plan(shift_overrides)
        )
        candidate_output, row = benchmark_candidate(
            name,
            lambda: apply_plan(
                candidate_plan,
                mode=mode,
                group_size=group_size,
                workers=workers,
            ),
            reference=reference_links,
            warmups=PROFILE["warmups"],
            repeats=PROFILE["repeats"],
            m_start=1,
            metadata={
                "plan_build_s": plan_build_s,
                "plan_bytes": candidate_plan.shift_plan.device_plan_memory_bytes,
                "row_chunk_size": int(shift_overrides.get("row_chunk_size", BASE_ROW_CHUNK_SIZE)),
                "lag_block_size": int(shift_overrides.get("lag_block_size", BASE_LAG_BLOCK_SIZE)),
                "group_size": int(group_size),
                "workers": int(workers),
                "execution_mode": mode,
                "shift_overrides": dict(shift_overrides),
            },
        )
        row["passes_numerics"] = numerical_gate(
            row,
            rel_l2_tolerance=REL_L2_TOLERANCE,
            max_abs_tolerance=MAX_ABS_TOLERANCE,
        )
        return row
    finally:
        del candidate_output
        if candidate_plan is not None:
            candidate_plan.clear_device_cache()
        del candidate_plan
        gc.collect()


## 8. Stage 1 — row-chunk sweep

Lag blocking and scheduling are held fixed. The materialized six-job path isolates the compiled assembly kernel from Python link scheduling.

In [ ]:
row_sweep = []
for row_size in PROFILE["row_sizes"]:
    print(f"Row chunk {row_size:,}")
    row_sweep.append(
        evaluate_plan_candidate(
            name=f"row_{row_size}",
            shift_overrides={
                "row_chunk_size": int(row_size),
                "lag_block_size": BASE_LAG_BLOCK_SIZE,
            },
        )
    )

best_row_result = fastest_passing(
    row_sweep,
    rel_l2_tolerance=REL_L2_TOLERANCE,
    max_abs_tolerance=MAX_ABS_TOLERANCE,
)
BEST_ROW_CHUNK_SIZE = int(best_row_result["row_chunk_size"])

row_sweep_table = pd.DataFrame(row_sweep)
display(
    row_sweep_table[
        [
            "name",
            "row_chunk_size",
            "first_call_s",
            "median_s",
            "iqr_s",
            "max_abs",
            "rel_l2",
            "passes_numerics",
            "rss_after_gib",
        ]
    ].sort_values("median_s")
)
print("Selected row_chunk_size:", BEST_ROW_CHUNK_SIZE)


## 9. Stage 2 — lag-block sweep

This changes only how the same 51 lags are blocked. It does **not** change $L_{\rm trunc}$.

In [ ]:
lag_sweep = []
for lag_size in PROFILE["lag_sizes"]:
    print(f"Lag block {lag_size}")
    lag_sweep.append(
        evaluate_plan_candidate(
            name=f"row_{BEST_ROW_CHUNK_SIZE}_lag_{lag_size}",
            shift_overrides={
                "row_chunk_size": BEST_ROW_CHUNK_SIZE,
                "lag_block_size": int(lag_size),
            },
        )
    )

best_lag_result = fastest_passing(
    lag_sweep,
    rel_l2_tolerance=REL_L2_TOLERANCE,
    max_abs_tolerance=MAX_ABS_TOLERANCE,
)
BEST_LAG_BLOCK_SIZE = int(best_lag_result["lag_block_size"])

lag_sweep_table = pd.DataFrame(lag_sweep)
display(
    lag_sweep_table[
        [
            "name",
            "row_chunk_size",
            "lag_block_size",
            "first_call_s",
            "median_s",
            "iqr_s",
            "max_abs",
            "rel_l2",
            "passes_numerics",
        ]
    ].sort_values("median_s")
)
print("Selected lag_block_size:", BEST_LAG_BLOCK_SIZE)


## 10. Stage 3 — link-schedule sweep

The winning row/lag configuration is built once. This stage chooses how the six directed links are divided across CPU workers.

In [ ]:
tuned_shift_overrides = {
    "row_chunk_size": BEST_ROW_CHUNK_SIZE,
    "lag_block_size": BEST_LAG_BLOCK_SIZE,
}
tuned_matches_baseline = (
    BEST_ROW_CHUNK_SIZE == BASE_ROW_CHUNK_SIZE
    and BEST_LAG_BLOCK_SIZE == BASE_LAG_BLOCK_SIZE
)
if tuned_matches_baseline:
    tuned_plan = baseline_plan
    tuned_plan_build_s = baseline_plan_build_s
else:
    tuned_plan, tuned_plan_build_s = timed_call(
        lambda: build_one_arm_plan(tuned_shift_overrides)
    )

schedule_sweep = []
for schedule in SCHEDULES:
    print(schedule["name"])
    schedule_output, schedule_row = benchmark_candidate(
        schedule["name"],
        lambda schedule=schedule: apply_plan(
            tuned_plan,
            mode=schedule["mode"],
            group_size=schedule["group_size"],
            workers=schedule["workers"],
        ),
        reference=reference_links,
        warmups=PROFILE["warmups"],
        repeats=PROFILE["repeats"],
        m_start=1,
        metadata={
            "row_chunk_size": BEST_ROW_CHUNK_SIZE,
            "lag_block_size": BEST_LAG_BLOCK_SIZE,
            "group_size": schedule["group_size"],
            "workers": schedule["workers"],
            "execution_mode": schedule["mode"],
        },
    )
    schedule_row["passes_numerics"] = numerical_gate(
        schedule_row,
        rel_l2_tolerance=REL_L2_TOLERANCE,
        max_abs_tolerance=MAX_ABS_TOLERANCE,
    )
    schedule_row["speedup_vs_current_selected"] = (
        baseline_shift_row["median_s"] / schedule_row["median_s"]
    )
    schedule_sweep.append(schedule_row)
    del schedule_output
    gc.collect()

best_schedule = fastest_passing(
    schedule_sweep,
    rel_l2_tolerance=REL_L2_TOLERANCE,
    max_abs_tolerance=MAX_ABS_TOLERANCE,
)

schedule_sweep_table = pd.DataFrame(schedule_sweep)
display(
    schedule_sweep_table[
        [
            "name",
            "group_size",
            "workers",
            "first_call_s",
            "median_s",
            "iqr_s",
            "speedup_vs_current_selected",
            "max_abs",
            "rel_l2",
            "passes_numerics",
        ]
    ].sort_values("median_s")
)
print("Selected schedule:", best_schedule["name"])


## 11. Optional new-backend experiments

After implementing reordered kernels, cached phases, factored phases, a real split kernel, compact interpolation, boundary specialisation, or frequency tiling, add one configuration in `CUSTOM_EXPERIMENTS`. Keep each refactor isolated until it passes this notebook.

In [ ]:
custom_results = []
for experiment in CUSTOM_EXPERIMENTS:
    custom_results.append(
        evaluate_plan_candidate(
            name=experiment["name"],
            shift_overrides=experiment["shift_overrides"],
            mode=experiment.get("mode", "materialized"),
            group_size=experiment.get("group_size", 6),
            workers=experiment.get("workers", 1),
        )
    )

if custom_results:
    custom_table = pd.DataFrame(custom_results)
    display(
        custom_table[
            [
                "name",
                "median_s",
                "iqr_s",
                "max_abs",
                "rel_l2",
                "passes_numerics",
                "plan_bytes",
            ]
        ].sort_values("median_s")
    )
else:
    print("No custom backend experiments configured.")


## 12. Complete A1 confirmation

This is the relevant end-to-end code benchmark: two parallel DWTs followed by all six one-arm responses. Scientific mismatch against the time-domain response remains a separate validation step.

In [ ]:
a1_results = {}
if RUN_COMPLETE_A1:
    def current_a1():
        context = parallel_dwt()
        return apply_plan(
            baseline_plan,
            mode="materialized_parallel_links",
            group_size=BASE_GROUP_SIZE,
            workers=BASE_WORKERS,
            plus=context["w_t_plus"],
            cross=context["w_t_cross"],
        )

    def tuned_a1():
        context = parallel_dwt()
        return apply_plan(
            tuned_plan,
            mode=best_schedule["execution_mode"],
            group_size=best_schedule["group_size"],
            workers=best_schedule["workers"],
            plus=context["w_t_plus"],
            cross=context["w_t_cross"],
        )

    baseline_a1_output, baseline_a1_row = benchmark_candidate(
        "current_a1",
        current_a1,
        warmups=PROFILE["warmups"],
        repeats=PROFILE["a1_repeats"],
        m_start=1,
    )
    tuned_a1_output, tuned_a1_row = benchmark_candidate(
        "tuned_a1",
        tuned_a1,
        reference=baseline_a1_output,
        warmups=PROFILE["warmups"],
        repeats=PROFILE["a1_repeats"],
        m_start=1,
    )
    tuned_a1_row["passes_numerics"] = numerical_gate(
        tuned_a1_row,
        rel_l2_tolerance=REL_L2_TOLERANCE,
        max_abs_tolerance=MAX_ABS_TOLERANCE,
    )
    tuned_a1_row["speedup_vs_current_a1"] = (
        baseline_a1_row["median_s"] / tuned_a1_row["median_s"]
    )
    a1_results = {"current": baseline_a1_row, "tuned": tuned_a1_row}
    display(
        pd.DataFrame([baseline_a1_row, tuned_a1_row])[
            ["name", "first_call_s", "median_s", "iqr_s", "max_abs", "rel_l2"]
        ]
    )
    print("A1 speedup:", tuned_a1_row["speedup_vs_current_a1"])
    del baseline_a1_output, tuned_a1_output
    gc.collect()
else:
    print("Complete A1 benchmark disabled.")


## 13. Decision gate and export

A tuning winner is recorded even if the gain is small, but it is marked `meaningful_speedup=False` unless it is at least 5% faster than the frozen selected baseline. New code refactors should generally be retained only if they clear that threshold or deliver a material memory reduction.

In [ ]:
selected_shift_speedup = (
    baseline_shift_row["median_s"] / best_schedule["median_s"]
)
meaningful_speedup = selected_shift_speedup >= MIN_MEANINGFUL_SPEEDUP

selection = {
    "row_chunk_size": BEST_ROW_CHUNK_SIZE,
    "lag_block_size": BEST_LAG_BLOCK_SIZE,
    "schedule_name": best_schedule["name"],
    "execution_mode": best_schedule["execution_mode"],
    "group_size": int(best_schedule["group_size"]),
    "workers": int(best_schedule["workers"]),
    "shift_speedup_vs_current_selected": float(selected_shift_speedup),
    "meaningful_speedup": bool(meaningful_speedup),
    "passes_numerics": bool(best_schedule["passes_numerics"]),
}

results = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_mode": RUN_MODE,
    "host": HOST,
    "source": {"WDM": WDM_STATE, "WDM_LISAresponse": RESPONSE_STATE},
    "provenance_ok": provenance_ok,
    "configuration": {
        "N": N,
        "dt": DT,
        "Nf": NF,
        "q": Q_WDM,
        "d": D_WDM,
        "A_frac": A_FRAC,
        "L_trunc": L_TRUNC,
        "calc_m0": CALC_M0,
        "baseline_shift_kwargs": BASE_SHIFT_KWARGS,
        "rel_l2_tolerance": REL_L2_TOLERANCE,
        "max_abs_tolerance": MAX_ABS_TOLERANCE,
    },
    "cache": {
        "waveform": WAVEFORM_CACHE,
        "wdm_coefficients": WDM_CACHE,
    },
    "setup_s": {
        "waveform_generation": waveform_generation_s,
        "grid": grid_build_s,
        "baseline_plan": baseline_plan_build_s,
        "tuned_plan": tuned_plan_build_s,
    },
    "dwt": dwt_results,
    "current_selected_shift": baseline_shift_row,
    "row_sweep": row_sweep,
    "lag_sweep": lag_sweep,
    "schedule_sweep": schedule_sweep,
    "custom_experiments": custom_results,
    "complete_a1": a1_results,
    "selection": selection,
    "final_memory": memory_snapshot(),
}

print(json.dumps(selection, indent=2))

if SAVE_RESULTS:
    RESULTS_DIR = BENCHMARK_DIR / "runtime_results"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    node = (HOST.get("node") or "unknown").replace("/", "_")
    result_stem = f"dense_runtime_{node}_{RUN_MODE}_{timestamp}"
    json_path = save_json(RESULTS_DIR / f"{result_stem}.json", results)

    flat_rows = []
    for stage, rows in (
        ("row", row_sweep),
        ("lag", lag_sweep),
        ("schedule", schedule_sweep),
        ("custom", custom_results),
    ):
        for row in rows:
            flat_rows.append(
                {
                    "stage": stage,
                    **{
                        key: value
                        for key, value in row.items()
                        if key not in {"all_s", "per_link", "shift_overrides"}
                    },
                }
            )
    csv_path = RESULTS_DIR / f"{result_stem}.csv"
    pd.DataFrame(flat_rows).to_csv(csv_path, index=False)
    print("Saved:", json_path)
    print("Saved:", csv_path)


## 14. Promotion checklist

Before changing the production defaults:

1. Set `RUN_MODE="full"` and restart the kernel.
2. Run the current baseline first, then the candidate; restart and reverse the order.
3. Confirm that the medians/IQRs agree across the two fresh processes.
4. Require the numerical gate, complete-A1 gate, and package tests to pass.
5. Run the winning implementation through the existing full response/mismatch notebook.
6. Commit one optimisation at a time and make the winner the next frozen baseline.

Do not infer isolated candidate peak memory from the cumulative process high-water mark. If memory is the claim, rerun only the baseline or only the candidate in a fresh process.